In [ ]:
#ORIGINal

def _get_bus_results(net, ppc_0, ppc_1, ppc_2, bus):
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]

    ppc_sequence = {0: ppc_0, 1: ppc_1, 2: ppc_2, "": ppc_1}
    if net["_options"]["fault"] == "LG":
        net.res_bus_sc["ikss_ka"] = ppc_0["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_0["bus"][ppc_index, SKSS]
        sequence_relevant = range(3)
    elif net["_options"]["fault"] == "LLG":
        sequence_relevant = range(3)
    elif net["_options"]["fault"] == "LL":
        net.res_bus_sc["ikss_ka"] = ppc_1["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_1["bus"][ppc_index, SKSS]
        sequence_relevant = range(3)
    else:
        net.res_bus_sc["ikss_ka"] = ppc_1["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_1["bus"][ppc_index, SKSS]
        sequence_relevant = ("",)
    for sequence in sequence_relevant:
        ppc_s = ppc_sequence[sequence]
        if net["_options"]["fault"] == "LL":
            # TODO: ask Marco where this can be done before results are written into tables
            if sequence in [1, 2]:
                fault_ohm_factor = 2
            elif sequence == 0:
                fault_ohm_factor = 1
            net.res_bus_sc[f"rk{sequence}_ohm"] = (ppc_s["bus"][ppc_index, R_EQUIV_OHM] +
                                                   net["_options"]["r_fault_ohm"]/fault_ohm_factor)
            net.res_bus_sc[f"xk{sequence}_ohm"] = (ppc_s["bus"][ppc_index, X_EQUIV_OHM] +
                                                   net["_options"]["x_fault_ohm"]/fault_ohm_factor)
        else:
            net.res_bus_sc[f"rk{sequence}_ohm"] = ppc_s["bus"][ppc_index, R_EQUIV_OHM]
            net.res_bus_sc[f"xk{sequence}_ohm"] = ppc_s["bus"][ppc_index, X_EQUIV_OHM]
        # in trafo3w, we add very high numbers (1e10) as impedances to block current
        # here, we need to replace such high values by np.inf
        baseZ = ppc_s["bus"][ppc_index, BASE_KV] ** 2 / ppc_s["baseMVA"]
        net.res_bus_sc.loc[net.res_bus_sc[f"xk{sequence}_ohm"] / baseZ > 1e9, f"xk{sequence}_ohm"] = np.inf
        net.res_bus_sc.loc[net.res_bus_sc[f"rk{sequence}_ohm"] / baseZ > 1e9, f"rk{sequence}_ohm"] = np.inf
    if net._options["ip"]:
        net.res_bus_sc["ip_ka"] = ppc_1["bus"][ppc_index, IP]
    if net._options["ith"]:
        net.res_bus_sc["ith_ka"] = ppc_1["bus"][ppc_index, ITH]

    net.res_bus_sc = net.res_bus_sc.loc[bus, :]

In [ ]:
#VERSION 1.0
def _get_bus_results(net, ppc_0, ppc_1, ppc_2, bus):

    ppc_sequence = {0: ppc_0, 1: ppc_1, 2: ppc_2, "": ppc_1}

    fault = net["_options"]["fault"]  # LG / LL / LLG / LLL

    # -------------- STANDARD PANDAPOWER PART -------------------
    if fault == "LG":
        net.res_bus_sc["ikss_ka"] = ppc_0["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_0["bus"][ppc_index, SKSS]
        sequence_relevant = range(3)

    elif fault == "LLG":
        sequence_relevant = range(3)

    elif fault == "LL":
        net.res_bus_sc["ikss_ka"] = ppc_1["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_1["bus"][ppc_index, SKSS]
        sequence_relevant = range(3)

    else:  # LLL (three-phase)
        net.res_bus_sc["ikss_ka"] = ppc_1["bus"][ppc_index, IKSSV] + ppc_1["bus"][ppc_index, IKSSC]
        net.res_bus_sc["skss_mw"] = ppc_1["bus"][ppc_index, SKSS]
        sequence_relevant = ("",)

    # ---- R/X impedance part stays same ----
    for sequence in sequence_relevant:
        ppc_s = ppc_sequence[sequence]

        if fault == "LL":
            fault_ohm_factor = 2 if sequence in [1, 2] else 1
            net.res_bus_sc[f"rk{sequence}_ohm"] = (ppc_s["bus"][ppc_index, R_EQUIV_OHM] +
                                                   net["_options"]["r_fault_ohm"] / fault_ohm_factor)
            net.res_bus_sc[f"xk{sequence}_ohm"] = (ppc_s["bus"][ppc_index, X_EQUIV_OHM] +
                                                   net["_options"]["x_fault_ohm"] / fault_ohm_factor)
        else:
            net.res_bus_sc[f"rk{sequence}_ohm"] = ppc_s["bus"][ppc_index, R_EQUIV_OHM]
            net.res_bus_sc[f"xk{sequence}_ohm"] = ppc_s["bus"][ppc_index, X_EQUIV_OHM]

        # infinity replacement
        baseZ = ppc_s["bus"][ppc_index, BASE_KV] ** 2 / ppc_s["baseMVA"]
        net.res_bus_sc.loc[net.res_bus_sc[f"xk{sequence}_ohm"] / baseZ > 1e9, f"xk{sequence}_ohm"] = np.inf
        net.res_bus_sc.loc[net.res_bus_sc[f"rk{sequence}_ohm"] / baseZ > 1e9, f"rk{sequence}_ohm"] = np.inf

    # standard ip / ith
    if net._options["ip"]:
        net.res_bus_sc["ip_ka"] = ppc_1["bus"][ppc_index, IP]
    if net._options["ith"]:
        net.res_bus_sc["ith_ka"] = ppc_1["bus"][ppc_index, ITH]

    # ------------- NEW PART: ABC STRUJE I SNAGE -----------------

    # extract sequence currents (complex)
    def seq_current(ppc, I_col, ANG_col):
        I = ppc["bus"][:, I_col]
        ang = np.deg2rad(ppc["bus"][:, ANG_col].real)
        return I * np.exp(1j * ang)

    i1_0 = seq_current(ppc_0, IKSSV, PHI_IKSSV_DEGREE)
    i1_1 = seq_current(ppc_1, IKSSV, PHI_IKSSV_DEGREE)
    i1_2 = seq_current(ppc_2, IKSSV, PHI_IKSSV_DEGREE)

    i2_0 = seq_current(ppc_0, IKSSC, PHI_IKSSV_DEGREE)
    i2_1 = seq_current(ppc_1, IKSSC, PHI_IKSSV_DEGREE)
    i2_2 = seq_current(ppc_2, IKSSC, PHI_IKSSV_DEGREE)

    # stack as (N, 3) sequence vectors
    I1 = np.vstack([i1_0, i1_1, i1_2]).T
    I2 = np.vstack([i2_0, i2_1, i2_2]).T

    # total sequence currents
    Iseq = I1 + I2

    # ---------- sequence to phase conversion ----------
    # standard Fortescue matrix
    a = np.exp(1j * 2 * np.pi / 3)
    A = np.array([[1,     1,     1    ],
                  [1, a**2,     a    ],
                  [1,    a,   a**2   ]])

    Iabc = (A @ Iseq.T).T / 3   # (N,3) matrix

    # magnitudes only for requested buses:
    Iph = np.abs(Iabc[ppc_index[bus], :])

    # base voltage per bus
    baseV = ppc_1["bus"][ppc_index[bus], BASE_KV]

    # phase MVA
    Skss = (Iph * baseV[:, None] / np.sqrt(3))

    # write into table
    for phase, col_i, col_s in zip([0,1,2],
                                   ["ikss_a_ka","ikss_b_ka","ikss_c_ka"],
                                   ["skss_a_mw","skss_b_mw","skss_c_mw"]):
        net.res_bus_sc[col_i] = np.nan
        net.res_bus_sc[col_s] = np.nan
        for idx, b in enumerate(bus):
            net.res_bus_sc.at[b, col_i] = Iph[idx, phase]
            net.res_bus_sc.at[b, col_s] = Skss[idx, phase]

    # ------------- RESTRICT TO ONLY USED BUSES -------------
    net.res_bus_sc = net.res_bus_sc.loc[bus, :]










In [ ]:
#VERSION 1 - LLL
def _calculate_bus_results_lll(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a 3-phase (LLL) fault.
    Structure mirrors the _calculate_bus_results_llg function but forces sequence 0 and 2 = 0.
    Writes results into net.res_bus_sc: ikss_a_ka, ikss_b_ka, ikss_c_ka and skss_a_mw, ...
    """

    # keep same lookup pattern as other functions
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]  # maps pandapower bus index -> ppc row

    n_ppc = len(ppc_index)

    # initialize full result matrices for all ppc buses (rows = ppc rows, cols = phases)
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) rotating machine (voltage source) contributions
    #    For LLL: seq 0 = 0, seq 1 = from ppc_1, seq 2 = 0
    # -----------------------------
    # ensure shapes are (n_ppc, 1)
    # use ppc_1 shape as reference (it must exist for LLL)
    ref_shape = ppc_1['bus'][:, IKSSV].shape[0]

    i_1_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_1_ka_1 = (ppc_1['bus'][:, IKSSV] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_1_ka_2 = np.zeros((ref_shape, 1), dtype=complex)

    # -----------------------------
    # 2) inverter-based generation (current source) contributions
    #    same: seq 0 = 0, seq 1 = from ppc_1, seq 2 = 0
    # -----------------------------
    i_2_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_2_ka_1 = (ppc_1['bus'][:, IKSSC] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_2_ka_2 = np.zeros((ref_shape, 1), dtype=complex)

    # -----------------------------
    # 3) stack sequences into shape compatible with sequence_to_phase call
    #    Following the LLG style: stack(..., axis=2) -> shape (n_ppc, 1, 3)
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], axis=2)  # (n_ppc, 1, 3)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], axis=2)  # (n_ppc, 1, 3)

    # -----------------------------
    # 4) transform sequence components to phase components
    #    apply_along_axis with axis=2 (length-3 vector of seq0,seq1,seq2)
    #    result shape: (n_ppc, 1, 3) -> squeeze to (n_ppc, 3)
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka)

    # squeeze the middle dim (the "from/to" dim which is 1 here)
    i_1_abc_ka = np.squeeze(i_1_abc_ka, axis=1)  # shape (n_ppc, 3)
    i_2_abc_ka = np.squeeze(i_2_abc_ka, axis=1)  # shape (n_ppc, 3)

    # zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # total phase currents (complex) per ppc row
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka  # shape (n_ppc, 3)

    # -----------------------------
    # 5) select the rows corresponding to the requested bus list
    #    bus is list/array of pandapower bus indices. Convert to ppc rows.
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]  # rows in ppc arrays corresponding to requested buses

    # absolute values and angles per phase for the requested buses
    # shape -> (len(bus), 3)
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, :]  # Keep complex values
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) voltages: build V_012 per requested buses then transform to phases
    #    For LLL: V0 = 0, V1 from ppc_1 internal, V2 = 0
    # -----------------------------
    # internal V_ikss in ppc_1 is indexed by ppc rows; use ppc_indices
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_0 = np.zeros_like(v_pu_1)
    v_pu_2 = np.zeros_like(v_pu_1)

    # stack into shape (len(bus), 1, 3) to reuse sequence_to_phase
    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], axis=2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)  # (len(bus),1,3) or more dims

    # Force to correct shape: squeeze all extra dimensions and reshape to (len(bus), 3)
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) compute short-circuit apparent power per phase (MVA) for selected buses
    #    baseV: use ppc_1 bus base_kv for the selected ppc_indices
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]  # (len(bus),) - 1D array

    # Reshape baseV to (len(bus), 1) for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, None]  # (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)  # (len(bus),3)

    # -----------------------------
    # 8) compute active and reactive power per phase (MW, Mvar)
    #    S = V * I* (conjugate)
    #    P = Re(S), Q = Im(S)
    # -----------------------------
    # Convert voltage to kV for power calculation
    # Phase voltage in kV - ensure proper broadcasting
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))  # shape (len(bus), 3)

    # Complex power per phase: S = V * conj(I)
    # Element-wise multiplication for each phase
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)  # (len(bus), 3)

    # Extract real and imaginary parts
    p_abc_mw_phase = np.real(s_abc_mva)  # Active power in MW
    q_abc_mvar_phase = np.imag(s_abc_mva)  # Reactive power in Mvar

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) fill the full result matrices at rows corresponding to ppc_indices
    #    use np.ix_ to assign submatrix
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs / 3
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    # compute per-phase voltages (pu and degree)
    v_abc_abs = np.full((n_ppc, 3), np.nan)
    v_abc_deg = np.full((n_ppc, 3), np.nan)

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) write results into net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        # Current magnitude and angle
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]

        # Apparent power
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]

        # Active and reactive power
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

        # Voltage magnitude and angle
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]

In [ ]:
#llg case - origigi
def _calculate_bus_results_lg(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a single line-to-ground (LG) fault.
    For LG fault: I0 = I1 = I2 (all sequence currents are equal)
    Writes results into net.res_bus_sc: ikss_a_ka, ikss_b_ka, ikss_c_ka and skss_a_mva, ...
    """

    # keep same lookup pattern as other functions
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]  # maps pandapower bus index -> ppc row

    n_ppc = len(ppc_index)

    # initialize full result matrices for all ppc buses (rows = ppc rows, cols = phases)
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) rotating machine (voltage source) contributions
    #    For LG: I0 = I1 = I2 (all sequence currents equal)
    # -----------------------------
    # ensure shapes are (n_ppc, 1)
    # use ppc_1 shape as reference (it must exist for LG)
    ref_shape = ppc_1['bus'][:, IKSSV].shape[0]

    # For LG fault, all sequence currents are equal
    i_1_ka_1 = (ppc_1['bus'][:, IKSSV] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_1_ka_0 = i_1_ka_1.copy()  # I0 = I1
    i_1_ka_2 = i_1_ka_1.copy()  # I2 = I1

    # -----------------------------
    # 2) inverter-based generation (current source) contributions
    #    same: I0 = I1 = I2
    # -----------------------------
    i_2_ka_1 = (ppc_1['bus'][:, IKSSC] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_2_ka_0 = i_2_ka_1.copy()  # I0 = I1
    i_2_ka_2 = i_2_ka_1.copy()  # I2 = I1

    # -----------------------------
    # 3) stack sequences into shape compatible with sequence_to_phase call
    #    Following the LLG style: stack(..., axis=2) -> shape (n_ppc, 1, 3)
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], axis=2)  # (n_ppc, 1, 3)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], axis=2)  # (n_ppc, 1, 3)

    # -----------------------------
    # 4) transform sequence components to phase components
    #    apply_along_axis with axis=2 (length-3 vector of seq0,seq1,seq2)
    #    result shape: (n_ppc, 1, 3) -> squeeze to (n_ppc, 3)
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka)

    # squeeze the middle dim (the "from/to" dim which is 1 here)
    i_1_abc_ka = np.squeeze(i_1_abc_ka, axis=1)  # shape (n_ppc, 3)
    i_2_abc_ka = np.squeeze(i_2_abc_ka, axis=1)  # shape (n_ppc, 3)

    # zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # total phase currents (complex) per ppc row
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka  # shape (n_ppc, 3)

    # -----------------------------
    # 5) select the rows corresponding to the requested bus list
    #    bus is list/array of pandapower bus indices. Convert to ppc rows.
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]  # rows in ppc arrays corresponding to requested buses

    # absolute values and angles per phase for the requested buses
    # shape -> (len(bus), 3)
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, :] / 3  # Keep complex values
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) voltages: build V_012 per requested buses then transform to phases
    #    For LG: V0 from ppc_0, V1 from ppc_1, V2 from ppc_2
    # -----------------------------
    # internal V_ikss in ppc_0, ppc_1 and ppc_2 is indexed by ppc rows; use ppc_indices
    v_pu_0 = ppc_0["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_2 = ppc_2["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)

    # stack into shape (len(bus), 1, 3) to reuse sequence_to_phase
    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], axis=2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)  # (len(bus),1,3) or more dims

    # Force to correct shape: squeeze all extra dimensions and reshape to (len(bus), 3)
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) compute short-circuit apparent power per phase (MVA) for selected buses
    #    baseV: use ppc_1 bus base_kv for the selected ppc_indices
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]  # (len(bus),) - 1D array

    # Reshape baseV to (len(bus), 1) for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, None]  # (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)  # (len(bus),3)

    # -----------------------------
    # 8) compute active and reactive power per phase (MW, Mvar)
    #    S = V * I* (conjugate)
    #    P = Re(S), Q = Im(S)
    # -----------------------------
    # Convert voltage to kV for power calculation
    # Phase voltage in kV - ensure proper broadcasting
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))  # shape (len(bus), 3)

    # Complex power per phase: S = V * conj(I)
    # Element-wise multiplication for each phase
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)  # (len(bus), 3)

    # Extract real and imaginary parts
    p_abc_mw_phase = np.real(s_abc_mva)  # Active power in MW
    q_abc_mvar_phase = np.imag(s_abc_mva)  # Reactive power in Mvar

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) fill the full result matrices at rows corresponding to ppc_indices
    #    use np.ix_ to assign submatrix
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    # compute per-phase voltages (pu and degree)
    v_abc_abs = np.full((n_ppc, 3), np.nan)
    v_abc_deg = np.full((n_ppc, 3), np.nan)

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) write results into net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        # Current magnitude and angle
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]

        # Apparent power
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]
        # Active and reactive power
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

        # Voltage magnitude and angle
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]

In [ ]:
def _calculate_bus_results_llg(ppc_0, ppc_1, ppc_2, bus, net):
    # we use 3D arrays here to easily identify via axis:
    # 0: line index, 1: from/to, 2: phase
    # short-ciruit for rotating machine (ext-grid and gen)
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]
    skss_abc_mva = np.full((len(ppc_index), 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((len(ppc_index), 3), np.nan, dtype=np.float64)

    i_1_ka_0 = ppc_0['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_1 = ppc_1['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_2 = ppc_2['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    # TODO check results with sgen
    # short-ciruit for inverter-based generation (current source)
    i_2_ka_0 = ppc_0['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_1 = ppc_1['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_2 = ppc_2['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], 2)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], 2)

    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka)

    # i_abc_ka = sequence_to_phase(np.vstack([i_ka_0, i_ka_1, i_ka_2]))
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-5] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-5] = 0

    # ToDo: check if this works without sgen
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka

    # Todo adapt to new reult format
    # Initialize a new matrix to store the selected rows
    # The shape is determined by the length of 'bus' and the number of columns in 'i_1_abc_ka'
    i_total_abc_ka_abs = np.zeros((len(bus), i_total_abc_ka.shape[2]))

    # Extract the specified rows from 'i_1_abc_ka' based on the indices in 'bus'
    # for index in range(len(bus)):
    #     i_total_abc_ka_abs[index] = abs(i_total_abc_ka[bus[index], bus[index]])
    ppc_indices = [ppc_index[b] for b in bus]
    i_total_abc_ka_abs = abs(i_total_abc_ka[ppc_indices, ppc_indices])

    # ToDo: check voltages
    v_pu_0 = ppc_0["internal"]["V_ikss"][bus][:, np.newaxis]
    v_pu_1 = ppc_1["internal"]["V_ikss"][bus][:, np.newaxis]
    v_pu_2 = ppc_2["internal"]["V_ikss"][bus][:, np.newaxis]

    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], 2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)
    # v_abc_pu = sequence_to_phase(np.vstack([v_pu_0, v_pu_1, v_pu_2]))
    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # this is inefficient because it copies data to fit into a shape, better to use a slice,
    # and even better to find how to use sequence-based powers:
    baseV = ppc_1['bus'][bus, BASE_KV][:, np.newaxis]
    # baseV = ppc_1["internal"]["baseV"][bus][:, np.newaxis]
    # v_base_kv = np.stack([baseV, baseV, baseV], 2)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)
    skss_abc_mva[np.ix_(bus, [0,1,2,])] = skss_abc_mva_phase
    ikss_abc_ka[np.ix_(bus, [0,1,2,])] = i_total_abc_ka_abs

    # Adding the ikss and skss values
    for i, phase in enumerate(['a', 'b', 'c']):
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]  # ikss values
        net.res_bus_sc[f'skss_{phase}_mw'] = skss_abc_mva[:, i]  # skss values

In [ ]:
#SVE CETIR ODVOJENO
def _calculate_bus_results_lll(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a 3-phase (LLL) fault.
    Structure mirrors the _calculate_bus_results_llg function but forces sequence 0 and 2 = 0.
    Writes results into net.res_bus_sc: ikss_a_ka, ikss_b_ka, ikss_c_ka and skss_a_mw, ...
    """

    # keep same lookup pattern as other functions
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]  # maps pandapower bus index -> ppc row

    n_ppc = len(ppc_index)

    # initialize full result matrices for all ppc buses (rows = ppc rows, cols = phases)
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) rotating machine (voltage source) contributions
    #    For LLL: seq 0 = 0, seq 1 = from ppc_1, seq 2 = 0
    # -----------------------------
    # ensure shapes are (n_ppc, 1)
    # use ppc_1 shape as reference (it must exist for LLL)
    ref_shape = ppc_1['bus'][:, IKSSV].shape[0]

    i_1_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_1_ka_1 = (ppc_1['bus'][:, IKSSV] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_1_ka_2 = np.zeros((ref_shape, 1), dtype=complex)

    # -----------------------------
    # 2) inverter-based generation (current source) contributions
    #    same: seq 0 = 0, seq 1 = from ppc_1, seq 2 = 0
    # -----------------------------
    i_2_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_2_ka_1 = (ppc_1['bus'][:, IKSSC] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_2_ka_2 = np.zeros((ref_shape, 1), dtype=complex)

    # -----------------------------
    # 3) stack sequences into shape compatible with sequence_to_phase call
    #    Following the LLG style: stack(..., axis=2) -> shape (n_ppc, 1, 3)
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], axis=2)  # (n_ppc, 1, 3)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], axis=2)  # (n_ppc, 1, 3)

    # -----------------------------
    # 4) transform sequence components to phase components
    #    apply_along_axis with axis=2 (length-3 vector of seq0,seq1,seq2)
    #    result shape: (n_ppc, 1, 3) -> squeeze to (n_ppc, 3)
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka / 3)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka / 3)

    # squeeze the middle dim (the "from/to" dim which is 1 here)
    i_1_abc_ka = np.squeeze(i_1_abc_ka, axis=1)  # shape (n_ppc, 3)
    i_2_abc_ka = np.squeeze(i_2_abc_ka, axis=1)  # shape (n_ppc, 3)

    # zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # total phase currents (complex) per ppc row
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka  # shape (n_ppc, 3)

    # -----------------------------
    # 5) select the rows corresponding to the requested bus list
    #    bus is list/array of pandapower bus indices. Convert to ppc rows.
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]  # rows in ppc arrays corresponding to requested buses

    # absolute values and angles per phase for the requested buses
    # shape -> (len(bus), 3)
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, :]  # Keep complex values
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) voltages: build V_012 per requested buses then transform to phases
    #    For LLL: V0 = 0, V1 from ppc_1 internal, V2 = 0
    # -----------------------------
    # internal V_ikss in ppc_1 is indexed by ppc rows; use ppc_indices
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_0 = np.zeros_like(v_pu_1)
    v_pu_2 = np.zeros_like(v_pu_1)

    # stack into shape (len(bus), 1, 3) to reuse sequence_to_phase
    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], axis=2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)  # (len(bus),1,3) or more dims

    # Force to correct shape: squeeze all extra dimensions and reshape to (len(bus), 3)
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) compute short-circuit apparent power per phase (MVA) for selected buses
    #    baseV: use ppc_1 bus base_kv for the selected ppc_indices
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]  # (len(bus),) - 1D array

    # Reshape baseV to (len(bus), 1) for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, None]  # (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)  # (len(bus),3)

    # -----------------------------
    # 8) compute active and reactive power per phase (MW, Mvar)
    #    S = V * I* (conjugate)
    #    P = Re(S), Q = Im(S)
    # -----------------------------
    # Convert voltage to kV for power calculation
    # Phase voltage in kV - ensure proper broadcasting
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))  # shape (len(bus), 3)

    # Complex power per phase: S = V * conj(I)
    # Element-wise multiplication for each phase
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)  # (len(bus), 3)

    # Extract real and imaginary parts
    p_abc_mw_phase = np.real(s_abc_mva)  # Active power in MW
    q_abc_mvar_phase = np.imag(s_abc_mva)  # Reactive power in Mvar

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) fill the full result matrices at rows corresponding to ppc_indices
    #    use np.ix_ to assign submatrix
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    # compute per-phase voltages (pu and degree)
    v_abc_abs = np.full((n_ppc, 3), np.nan)
    v_abc_deg = np.full((n_ppc, 3), np.nan)

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) write results into net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        # Current magnitude and angle
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]

        # Apparent power
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]

        # Active and reactive power
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

        # Voltage magnitude and angle
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]
================================================================================================

def _calculate_bus_results_ll(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a line-to-line (LL) fault.
    For LL fault: sequence 0 = 0, sequence 1 = from ppc_1, sequence 2 = -sequence 1
    Writes results into net.res_bus_sc: ikss_a_ka, ikss_b_ka, ikss_c_ka and skss_a_mva, ...
    """

    # keep same lookup pattern as other functions
    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]  # maps pandapower bus index -> ppc row

    n_ppc = len(ppc_index)

    # initialize full result matrices for all ppc buses (rows = ppc rows, cols = phases)
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) rotating machine (voltage source) contributions
    #    For LL: seq 0 = 0, seq 1 = from ppc_1, seq 2 = -seq 1
    # -----------------------------
    # ensure shapes are (n_ppc, 1)
    # use ppc_1 shape as reference (it must exist for LL)
    ref_shape = ppc_1['bus'][:, IKSSV].shape[0]

    i_1_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_1_ka_1 = (ppc_1['bus'][:, IKSSV] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_1_ka_2 = -i_1_ka_1  # For LL fault: I2 = -I1

    # -----------------------------
    # 2) inverter-based generation (current source) contributions
    #    same: seq 0 = 0, seq 1 = from ppc_1, seq 2 = -seq 1
    # -----------------------------
    i_2_ka_0 = np.zeros((ref_shape, 1), dtype=complex)
    i_2_ka_1 = (ppc_1['bus'][:, IKSSC] *
                np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real)))[:, None]
    i_2_ka_2 = -i_2_ka_1  # For LL fault: I2 = -I1

    # -----------------------------
    # 3) stack sequences into shape compatible with sequence_to_phase call
    #    Following the LLG style: stack(..., axis=2) -> shape (n_ppc, 1, 3)
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], axis=2)  # (n_ppc, 1, 3)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], axis=2)  # (n_ppc, 1, 3)

    # -----------------------------
    # 4) transform sequence components to phase components
    #    apply_along_axis with axis=2 (length-3 vector of seq0,seq1,seq2)
    #    result shape: (n_ppc, 1, 3) -> squeeze to (n_ppc, 3)
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka / np.sqrt(3))
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka / np.sqrt(3))

    # squeeze the middle dim (the "from/to" dim which is 1 here)
    i_1_abc_ka = np.squeeze(i_1_abc_ka, axis=1)  # shape (n_ppc, 3)
    i_2_abc_ka = np.squeeze(i_2_abc_ka, axis=1)  # shape (n_ppc, 3)

    # zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # total phase currents (complex) per ppc row
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka  # shape (n_ppc, 3)

    # -----------------------------
    # 5) select the rows corresponding to the requested bus list
    #    bus is list/array of pandapower bus indices. Convert to ppc rows.
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]  # rows in ppc arrays corresponding to requested buses

    # absolute values and angles per phase for the requested buses
    # shape -> (len(bus), 3)
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, :]  # Keep complex values
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) voltages: build V_012 per requested buses then transform to phases
    #    For LL: V0 = 0, V1 from ppc_1 internal, V2 from ppc_2 internal
    # -----------------------------
    # internal V_ikss in ppc_1 and ppc_2 is indexed by ppc rows; use ppc_indices
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_2 = ppc_2["internal"]["V_ikss"][ppc_indices][:, None]  # shape (len(bus),1)
    v_pu_0 = np.zeros_like(v_pu_1)

    # stack into shape (len(bus), 1, 3) to reuse sequence_to_phase
    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], axis=2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)  # (len(bus),1,3) or more dims

    # Force to correct shape: squeeze all extra dimensions and reshape to (len(bus), 3)
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) compute short-circuit apparent power per phase (MVA) for selected buses
    #    baseV: use ppc_1 bus base_kv for the selected ppc_indices
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]  # (len(bus),) - 1D array

    # Reshape baseV to (len(bus), 1) for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, None]  # (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)  # (len(bus),3)

    # -----------------------------
    # 8) compute active and reactive power per phase (MW, Mvar)
    #    S = V * I* (conjugate)
    #    P = Re(S), Q = Im(S)
    # -----------------------------
    # Convert voltage to kV for power calculation
    # Phase voltage in kV - ensure proper broadcasting
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))  # shape (len(bus), 3)

    # Complex power per phase: S = V * conj(I)
    # Element-wise multiplication for each phase
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)  # (len(bus), 3)

    # Extract real and imaginary parts
    p_abc_mw_phase = np.real(s_abc_mva)  # Active power in MW
    q_abc_mvar_phase = np.imag(s_abc_mva)  # Reactive power in Mvar

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) fill the full result matrices at rows corresponding to ppc_indices
    #    use np.ix_ to assign submatrix
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    # compute per-phase voltages (pu and degree)
    v_abc_abs = np.full((n_ppc, 3), np.nan)
    v_abc_deg = np.full((n_ppc, 3), np.nan)

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) write results into net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        # Current magnitude and angle
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]

        # Apparent power
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]

        # Active and reactive power
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

        # Voltage magnitude and angle
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]

=================================================================================================================================================
def _calculate_bus_results_lg(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a single line-to-ground (LG) fault.
    For LG fault: I0 = I1 = I2 (all sequence currents are equal)
    Follows the structure from _calculate_bus_results_llg.
    """

    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]

    n_ppc = len(ppc_index)

    # Initialize result arrays for all ppc buses
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    v_abc_abs = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    v_abc_deg = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) Rotating machine (voltage source) contributions
    #    For LG: I0 = I1 = I2 (all sequence currents equal)
    # -----------------------------
    i_1_ka_0 = ppc_0['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_1 = ppc_1['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_2 = ppc_2['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    # -----------------------------
    # 2) Inverter-based generation (current source) contributions
    # -----------------------------
    i_2_ka_0 = ppc_0['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_1 = ppc_1['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_2 = ppc_2['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    # -----------------------------
    # 3) Stack sequence components
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], 2)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], 2)

    # -----------------------------
    # 4) Transform to phase components
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka / 3)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka / 3)

    # Zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # Total phase currents
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka

    # -----------------------------
    # 5) Select requested buses
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]

    # Extract currents for requested buses (diagonal elements for bus fault)
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, ppc_indices]  # shape (len(bus), 3)
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) Voltages: transform from sequence to phase
    #    For LG: use all three sequence voltages from ppc_0, ppc_1, ppc_2
    # -----------------------------
    v_pu_0 = ppc_0["internal"]["V_ikss"][ppc_indices][:, np.newaxis]
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, np.newaxis]
    v_pu_2 = ppc_2["internal"]["V_ikss"][ppc_indices][:, np.newaxis]

    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], 2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)

    # Force proper shape - squeeze and reshape if needed
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) Apparent power per phase
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]

    # Reshape baseV for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, np.newaxis]  # shape (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)

    # -----------------------------
    # 8) Active and reactive power per phase
    #    S = V * I* (conjugate)
    # -----------------------------
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)

    p_abc_mw_phase = np.real(s_abc_mva)
    q_abc_mvar_phase = np.imag(s_abc_mva)

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) Fill result matrices
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) Write results to net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

=======================================================================================================
def _calculate_bus_results_llg(ppc_0, ppc_1, ppc_2, bus, net):
    """
    Calculate per-phase short-circuit currents and powers for a double line-to-ground (LLG) fault.
    Writes results into net.res_bus_sc with all required columns.
    """
    # we use 3D arrays here to easily identify via axis:
    # 0: line index, 1: from/to, 2: phase

    bus_lookup = net._pd2ppc_lookups["bus"]
    ppc_index = bus_lookup[net.bus.index]

    n_ppc = len(ppc_index)

    # Initialize result arrays for all ppc buses
    skss_abc_mva = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_ka = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    ikss_abc_degree = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    p_abc_mw = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    q_abc_mvar = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    v_abc_abs = np.full((n_ppc, 3), np.nan, dtype=np.float64)
    v_abc_deg = np.full((n_ppc, 3), np.nan, dtype=np.float64)

    # -----------------------------
    # 1) Rotating machine (voltage source) contributions
    # -----------------------------
    i_1_ka_0 = ppc_0['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_1 = ppc_1['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_1_ka_2 = ppc_2['bus'][:, IKSSV] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    # -----------------------------
    # 2) Inverter-based generation (current source) contributions
    # -----------------------------
    i_2_ka_0 = ppc_0['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_0['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_1 = ppc_1['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_1['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]
    i_2_ka_2 = ppc_2['bus'][:, IKSSC] * np.exp(1j * np.deg2rad(ppc_2['bus'][:, PHI_IKSSV_DEGREE].real))[:, np.newaxis]

    # -----------------------------
    # 3) Stack sequence components
    # -----------------------------
    i_1_012_ka = np.stack([i_1_ka_0, i_1_ka_1, i_1_ka_2], 2)
    i_2_012_ka = np.stack([i_2_ka_0, i_2_ka_1, i_2_ka_2], 2)

    # -----------------------------
    # 4) Transform to phase components
    # -----------------------------
    i_1_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_1_012_ka)
    i_2_abc_ka = np.apply_along_axis(sequence_to_phase, 2, i_2_012_ka)

    # Zero small values
    i_1_abc_ka[np.abs(i_1_abc_ka) < 1e-6] = 0
    i_2_abc_ka[np.abs(i_2_abc_ka) < 1e-6] = 0

    # Total phase currents
    i_total_abc_ka = i_1_abc_ka + i_2_abc_ka

    # -----------------------------
    # 5) Select requested buses
    # -----------------------------
    ppc_indices = [ppc_index[b] for b in bus]

    # Extract currents for requested buses (diagonal elements for bus fault)
    # Keep complex values for angle calculation
    i_total_abc_ka_complex = i_total_abc_ka[ppc_indices, ppc_indices]  # shape (len(bus), 3)
    i_total_abc_ka_abs = np.abs(i_total_abc_ka_complex)
    i_total_abc_ka_angle = np.angle(i_total_abc_ka_complex, deg=True)

    # -----------------------------
    # 6) Voltages: transform from sequence to phase
    #    For LLG: use all three sequence voltages
    # -----------------------------
    v_pu_0 = ppc_0["internal"]["V_ikss"][ppc_indices][:, np.newaxis]
    v_pu_1 = ppc_1["internal"]["V_ikss"][ppc_indices][:, np.newaxis]
    v_pu_2 = ppc_2["internal"]["V_ikss"][ppc_indices][:, np.newaxis]

    v_012_pu = np.stack([v_pu_0, v_pu_1, v_pu_2], 2)
    v_abc_pu = np.apply_along_axis(sequence_to_phase, 2, v_012_pu)

    # Force proper shape - squeeze and reshape if needed
    v_abc_pu = np.squeeze(v_abc_pu)  # Remove all singleton dimensions
    if v_abc_pu.ndim == 1:  # Single bus case: (3,)
        v_abc_pu = v_abc_pu.reshape(1, 3)
    elif v_abc_pu.ndim > 2:  # Still has extra dimensions
        v_abc_pu = v_abc_pu.reshape(len(bus), 3)

    v_abc_pu[np.abs(v_abc_pu) < 1e-5] = 0

    # -----------------------------
    # 7) Apparent power per phase
    # -----------------------------
    baseV = ppc_1['bus'][ppc_indices, BASE_KV]

    # Reshape baseV for proper broadcasting
    if baseV.ndim == 1:
        baseV = baseV[:, np.newaxis]  # shape (len(bus), 1)

    skss_abc_mva_phase = i_total_abc_ka_abs * baseV / np.sqrt(3)

    # -----------------------------
    # 8) Active and reactive power per phase
    #    S = V * I* (conjugate)
    # -----------------------------
    v_abc_kv = v_abc_pu * (baseV / np.sqrt(3))
    s_abc_mva = v_abc_kv * np.conj(i_total_abc_ka_complex)

    p_abc_mw_phase = np.real(s_abc_mva)
    q_abc_mvar_phase = np.imag(s_abc_mva)

    # Zero out small values
    p_abc_mw_phase[np.abs(p_abc_mw_phase) < 1e-6] = 0
    q_abc_mvar_phase[np.abs(q_abc_mvar_phase) < 1e-6] = 0

    # -----------------------------
    # 9) Fill result matrices
    # -----------------------------
    ikss_abc_ka[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_abs
    ikss_abc_degree[np.ix_(ppc_indices, [0, 1, 2])] = i_total_abc_ka_angle
    skss_abc_mva[np.ix_(ppc_indices, [0, 1, 2])] = skss_abc_mva_phase
    p_abc_mw[np.ix_(ppc_indices, [0, 1, 2])] = p_abc_mw_phase
    q_abc_mvar[np.ix_(ppc_indices, [0, 1, 2])] = q_abc_mvar_phase

    v_abc_abs[np.ix_(ppc_indices, [0, 1, 2])] = np.abs(v_abc_pu)
    v_abc_deg[np.ix_(ppc_indices, [0, 1, 2])] = np.angle(v_abc_pu, deg=True)

    # -----------------------------
    # 10) Write results to net.res_bus_sc
    # -----------------------------
    for i, phase in enumerate(['a', 'b', 'c']):
        # Current magnitude and angle
        net.res_bus_sc[f'ikss_{phase}_ka'] = ikss_abc_ka[:, i]
        net.res_bus_sc[f'ikss_{phase}_degree'] = ikss_abc_degree[:, i]

        # Apparent power
        net.res_bus_sc[f'skss_{phase}_mva'] = skss_abc_mva[:, i]

        # Active and reactive power
        net.res_bus_sc[f'p_{phase}_mw'] = p_abc_mw[:, i]
        net.res_bus_sc[f'q_{phase}_mvar'] = q_abc_mvar[:, i]

        # Voltage magnitude and angle
        net.res_bus_sc[f'vm_{phase}_pu'] = v_abc_abs[:, i]
        net.res_bus_sc[f'va_{phase}_degree'] = v_abc_deg[:, i]